# APS360 Pose-Based Human Action Recognition


# Data Loading

In [1]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:

!pip install -q mpose

import os
import numpy as np

  Preparing metadata (setup.py) ... done


In [3]:
cache_dir = '/content/drive/MyDrive/APS360/Project/data'
os.makedirs(cache_dir, exist_ok=True)
npz_path = os.path.join(cache_dir, 'mpose_posenet_split1.npz')

if os.path.exists(npz_path):
    z = np.load(npz_path)
    X_train, y_train, X_test, y_test = z['X_train'], z['y_train'], z['X_test'], z['y_test']
    print('loaded from Drive cache')
else:
    import mpose
    dataset = mpose.MPOSE(pose_extractor='posenet', split=1)   # raw keypoints, do my own data preprocessing
    dataset.get_info()
    X_train, y_train, X_test, y_test = dataset.get_data()
    np.savez_compressed(npz_path,
                        X_train=X_train, y_train=y_train,
                        X_test=X_test, y_test=y_test)
    print('downloaded + cached to Drive')

print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)


loaded from Drive cache
(12562, 30, 17, 3) (12562,) (2867, 30, 17, 3) (2867,)


In [4]:
# sanity check before data prepocessing

# check the labels in the dataset
print("y dtype:", y_train.dtype)
print("unique labels:", np.unique(y_train))
print("num classes:", len(np.unique(y_train)))
print("first 10:", y_train[:10])

# check the percentage of each class
vals, counts = np.unique(y_train, return_counts=True)
print("\nclass distribution (train):")
for v, c in zip(vals, counts):
    print(f"  class {v}: {c:5d}  ({100*c/len(y_train):.1f}%)")
print(f"imbalance ratio (max/min): {counts.max()/counts.min():.1f}x")

# check the value range per channel, needed for sample normalization
for i, name in enumerate(['x', 'y', 'conf']):
    ch = X_train[..., i]
    print(f"\n{name}: min={ch.min():.3f}  max={ch.max():.3f}  mean={ch.mean():.3f}")


y dtype: int64
unique labels: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]
num classes: 20
first 10: [13 13 13 13 13 13 13 13 13 13]

class distribution (train):
  class 0:   426  (3.4%)
  class 1:   199  (1.6%)
  class 2:   288  (2.3%)
  class 3:   241  (1.9%)
  class 4:   298  (2.4%)
  class 5:   241  (1.9%)
  class 6:   227  (1.8%)
  class 7:  1872  (14.9%)
  class 8:  1070  (8.5%)
  class 9:  1537  (12.2%)
  class 10:   300  (2.4%)
  class 11:   419  (3.3%)
  class 12:   197  (1.6%)
  class 13:   369  (2.9%)
  class 14:  1441  (11.5%)
  class 15:  1826  (14.5%)
  class 16:   550  (4.4%)
  class 17:   204  (1.6%)
  class 18:   243  (1.9%)
  class 19:   614  (4.9%)
imbalance ratio (max/min): 9.5x

x: min=0.000  max=416.000  mean=200.448

y: min=0.000  max=288.000  mean=117.859

conf: min=0.000  max=0.997  mean=0.806


In [6]:

# keypoint indices
lhip, rhip, lshoulder, rshoulder = 11, 12, 5, 6
conf_thrhold= 0.2

def preprocess(X):
    """
    X: (N, 30, 17, 3) raw pixel keypoints (x, y, conf)
    -> (N, 30, 51) normalized + cleaned, flattened per frame
    """
    X = X.astype(np.float32).copy()   # make copy so not touch the original source data
    xy   = X[..., :2]    # (N,30,17,2)
    conf = X[..., 2:3]    # (N,30,17,1)

    # reference points, computed per frame
    mid_hip = (xy[:, :, lhip] + xy[:, :, rhip]) / 2          # (N,30,2)
    mid_sho = (xy[:, :, lshoulder] + xy[:, :, rshoulder]) / 2   # (N,30,2)
    torso   = np.linalg.norm(mid_sho - mid_hip, axis=-1)       # (N,30)
    torso   = np.clip(torso, 0.001, None)[..., None, None]      # avoid divide-by-zero

    # normalization
    # center at mid-hip, scale by torso length
    xy = (xy - mid_hip[:, :, None, :]) / torso

    # zero out low-confidence joints (position and confidence flag)
    bad = conf < conf_thrhold
    xy   = np.where(bad, 0.0, xy)
    conf = np.where(bad, 0.0, conf)

    #recombine + flatten
    X = np.concatenate([xy, conf], axis=-1)   # (N,30,17,3)
    return X.reshape(X.shape[0], X.shape[1], -1)  # (N,30,51)


In [7]:
X_train_p = preprocess(X_train)
X_test_p  = preprocess(X_test)
# sanity check; expect (12562, 30, 51) (2867, 30, 51)
print(X_train_p.shape, X_test_p.shape)

(12562, 30, 51) (2867, 30, 51)
